# Depthwise Separable Convolution（深度可分离卷积）— 面试版

**难度：** Medium

**面试要点：** 不使用 `unfold`，纯手写 im2col 实现卷积

In [ ]:
import torch
import math

In [ ]:
# ✅ INTERVIEW

def depthwise_separable_conv(x, dw_weight, pw_weight):
    B, C_in, H, W = x.shape
    kH, kW = dw_weight.shape[2], dw_weight.shape[3]
    H_out = H - kH + 1
    W_out = W - kW + 1

    # ---- Step 1: 手写 im2col 提取 patches ----
    # 面试高频考点：不依赖 unfold，手动构造滑窗索引
    # 用嵌套循环或高级索引提取每个位置的 kH×kW 窗口
    # 这里用列表推导 + stack 实现，面试时也可以写 for 循环
    patches = torch.zeros(B, C_in, H_out, W_out, kH, kW)
    for i in range(kH):
        for j in range(kW):
            # x[:, :, i:i+H_out, j:j+W_out] 取出偏移 (i,j) 处的切片
            # 形状: (B, C_in, H_out, W_out)
            # 赋值到 patches 的第4,5维对应位置
            patches[:, :, :, :, i, j] = x[:, :, i:i+H_out, j:j+W_out]

    # ---- Step 2: 深度卷积 — 逐通道卷积 ----
    # dw_weight[:, 0] 形状: (C_in, kH, kW)
    # view(1, C_in, 1, 1, kH, kW) 用于广播:
    #   dim0=1 → 广播到 B 个样本
    #   dim1=C_in → 每个通道用自己的卷积核
    #   dim2,3=1 → 广播到所有空间位置
    # 逐元素乘法后对最后两维求和 = 卷积操作
    dw_out = (patches * dw_weight[:, 0].view(1, C_in, 1, 1, kH, kW)).sum(dim=(-2, -1))

    # ---- Step 3: 逐点卷积 — 手写矩阵乘法 ----
    # 不用 einsum，用 reshape + matmul 实现
    # dw_out: (B, C_in, H_out, W_out)
    # 我们需要对每个空间位置做: out[b, o, i, j] = sum_c weight[o, c] * dw_out[b, c, i, j]
    # 步骤:
    #   dw_out.reshape(B, C_in, -1) → (B, C_in, H_out*W_out)
    #   pw_weight[:, :, 0, 0].T → (C_in, C_out)
    #   矩阵乘法 → (B, C_out, H_out*W_out)
    #   reshape回 → (B, C_out, H_out, W_out)
    pw = pw_weight[:, :, 0, 0]  # (C_out, C_in)
    out = torch.matmul(pw, dw_out.reshape(B, C_in, -1)).reshape(B, -1, H_out, W_out)
    return out

In [ ]:
torch.manual_seed(0)
B, C_in, H, W = 2, 4, 8, 8
C_out, kH, kW = 8, 3, 3

x         = torch.randn(B, C_in, H, W)
dw_weight = torch.randn(C_in, 1, kH, kW)
pw_weight = torch.randn(C_out, C_in, 1, 1)

out = depthwise_separable_conv(x, dw_weight, pw_weight)
print("Output shape:", out.shape)  # (2, 8, 6, 6)

In [ ]:
from torch_judge import check
check("depthwise_conv")

## 📝 核心思路总结

1. **im2col 手写**：用双层 for 循环提取滑窗 patches，面试常考
2. **深度卷积**：每个通道独立卷积，权重形状 (C_in, 1, kH, kW) 的 "1" 是关键
3. **逐点卷积**：本质是矩阵乘法，reshape 后用 `matmul` 实现
4. **数值等价性**：两种实现应产生完全相同的结果